## Part 1: Parsing a Sentence with CKY

Given the following grammar in CNF.
1. Write the code for CKY algorithm and verify if the **sentence** follows or can be parsed by this grammar or not.
2. Also print the final CKY chart/table.

### How CKY works

CKY (Cocke–Kasami–Younger) is a **bottom-up dynamic-programming** recogniser/parser that requires the
grammar to be in **Chomsky Normal Form (CNF)**, i.e. every production is either

* `A -> B C` (exactly two non-terminals), or
* `A -> 'word'` (exactly one terminal).

For a sentence of `n` words we fill an upper-triangular chart where

> `chart[i][j]` = the set of non-terminals that can derive the span `words[i:j]`.

The algorithm has three stages:

1. **Lexical (diagonal) fill** — for every word `words[i]`, put every `A` with `A -> words[i]` into `chart[i][i+1]`.
2. **Binary fill** — for every span length `2 … n`, for every start `i`, and for every *split point* `k` with
   `i < k < j`, if `B ∈ chart[i][k]` and `C ∈ chart[k][j]` and the rule `A -> B C` exists, add `A` to `chart[i][j]`.
3. **Acceptance test** — the sentence is in the language iff the start symbol `S ∈ chart[0][n]`.

Complexity is `O(n³ · |G|)`: `n²` cells × `n` split points × the number of grammar rules.

To also *recover the trees* (not just answer yes/no) we store **back-pointers**: instead of only the label `A`,
each cell keeps, for every `A`, the list of `(k, B, C)` triples that produced it. Re-expanding those triples
recursively gives every parse tree.

In [1]:
import nltk
from nltk.parse import ChartParser
from nltk.grammar import CFG, Nonterminal, is_terminal, is_nonterminal
from nltk.tree import Tree

In [2]:
cky_grammar = CFG.fromstring("""
    S -> NP VP
    NP -> Det N
    VP -> V NP
    Det -> 'the'
    N -> 'man' | 'dog'
    V -> 'saw'
""")

sentence = "the man saw the dog"

### Step 1 — Read the grammar rules out of the NLTK `CFG` object

Rather than re-typing the grammar as a Python dictionary, we read the productions straight from
`cky_grammar`. This keeps a single source of truth, so the CKY parser and NLTK's parser are
guaranteed to use exactly the same grammar in Part 2.

While doing that we also **verify the grammar really is in CNF** — if any production is neither
`A -> B C` nor `A -> 'word'`, we raise an error, because CKY would silently give wrong answers.

In [3]:
def extract_cnf_rules(grammar):
    """Split a CNF grammar into binary rules and lexical rules.

    Returns
    -------
    binary  : list of (A, B, C)      for every rule  A -> B C
    lexical : dict word -> [A, ...]  for every rule  A -> 'word'
    """
    binary, lexical = [], {}

    for prod in grammar.productions():
        lhs = prod.lhs().symbol()
        rhs = prod.rhs()

        if len(rhs) == 1 and is_terminal(rhs[0]):          # A -> 'word'
            lexical.setdefault(rhs[0], []).append(lhs)
        elif len(rhs) == 2 and all(is_nonterminal(s) for s in rhs):   # A -> B C
            binary.append((lhs, rhs[0].symbol(), rhs[1].symbol()))
        else:
            raise ValueError(f"Production is not in CNF: {prod}")

    return binary, lexical


binary_rules, lexical_rules = extract_cnf_rules(cky_grammar)

print("Grammar is in CNF :", cky_grammar.is_chomsky_normal_form())
print("Start symbol      :", cky_grammar.start())
print()
print("Binary rules (A -> B C):")
for a, b, c in binary_rules:
    print(f"   {a} -> {b} {c}")
print()
print("Lexical rules (A -> 'word'):")
for word, tags in sorted(lexical_rules.items()):
    print(f"   {word:<10} -> {tags}")

Grammar is in CNF : True
Start symbol      : S

Binary rules (A -> B C):
   S -> NP VP
   NP -> Det N
   VP -> V NP

Lexical rules (A -> 'word'):
   dog        -> ['N']
   man        -> ['N']
   saw        -> ['V']
   the        -> ['Det']


### Step 2 — The CKY algorithm

`chart[i][j]` is a dictionary mapping a non-terminal to the list of ways it was derived:

* a **string** (the word itself) for a lexical entry, or
* a **`(k, B, C)` back-pointer** meaning "`A -> B C` with `B` covering `words[i:k]` and `C` covering `words[k:j]`".

Keeping *all* the derivations (instead of just the label) is what lets us print every parse tree later,
and it is also what makes ambiguity visible: a cell entry with two back-pointers means two different parses.

In [4]:
def cky_parse(words, grammar):
    """Run the CKY algorithm.

    Parameters
    ----------
    words   : list[str]  the tokenised sentence
    grammar : nltk CFG in Chomsky Normal Form

    Returns
    -------
    chart     : (n+1) x (n+1) table; chart[i][j] maps a non-terminal to its back-pointers
    accepted  : True iff the start symbol spans the whole sentence
    """
    binary, lexical = extract_cnf_rules(grammar)
    start = grammar.start().symbol()
    n = len(words)

    # chart[i][j] covers words[i:j]; only cells with j > i are ever used.
    chart = [[dict() for _ in range(n + 1)] for _ in range(n + 1)]

    # --- Stage 1: lexical fill (spans of length 1) -------------------------
    for i, word in enumerate(words):
        for lhs in lexical.get(word, []):
            chart[i][i + 1].setdefault(lhs, []).append(word)

    # --- Stage 2: binary fill (spans of length 2 .. n) ---------------------
    for length in range(2, n + 1):              # span width
        for i in range(0, n - length + 1):      # span start
            j = i + length                      # span end (exclusive)
            for k in range(i + 1, j):           # split point
                for lhs, b, c in binary:
                    if b in chart[i][k] and c in chart[k][j]:
                        chart[i][j].setdefault(lhs, []).append((k, b, c))

    # --- Stage 3: acceptance ----------------------------------------------
    accepted = start in chart[0][n]
    return chart, accepted

### Step 3 — Does the sentence belong to the grammar?

In [5]:
words = sentence.split()
chart, accepted = cky_parse(words, cky_grammar)

print("Sentence :", sentence)
print("Tokens   :", words)
print()
print(f"S in chart[0][{len(words)}] ?  ->  {accepted}")
print("The sentence CAN be parsed by this grammar."
      if accepted else
      "The sentence CANNOT be parsed by this grammar.")

Sentence : the man saw the dog
Tokens   : ['the', 'man', 'saw', 'the', 'dog']

S in chart[0][5] ?  ->  True
The sentence CAN be parsed by this grammar.


### Step 4 — Print the final CKY chart

Two views of the same table:

* a **grid** where row `i` / column `j` holds the labels covering `words[i:j]`
  (only the upper triangle is filled — a span must have `j > i`);
* a **span-by-span listing**, which is easier to read for longer sentences.

In [6]:
def print_chart_grid(chart, words):
    """Print the CKY chart as an upper-triangular grid."""
    n = len(words)
    cells = [[("{" + ", ".join(sorted(chart[i][j])) + "}" if chart[i][j] else "-")
              if j > i else ""
              for j in range(n + 1)]
             for i in range(n)]

    row_names = [f"i={i} ({words[i]})" for i in range(n)]
    head_w = max(len(r) for r in row_names) + 2
    width  = max([len(c) for row in cells for c in row] + [len(w) for w in words] + [6]) + 2

    head = lambda s: str(s).ljust(head_w)
    cell = lambda s: str(s).ljust(width)

    print("CKY chart -- chart[i][j] = labels spanning words[i:j]\n")
    print(head("") + "".join(cell(f"j={j}") for j in range(1, n + 1)))
    print(head("") + "".join(cell(w) for w in words))
    print("-" * (head_w + width * n))
    for i in range(n):
        print(head(row_names[i]) + "".join(cell(cells[i][j]) for j in range(1, n + 1)))


def print_chart_spans(chart, words):
    """Print every non-empty chart cell as 'span -> labels'."""
    n = len(words)
    print("\nNon-empty cells (by increasing span length):\n")
    for length in range(1, n + 1):
        for i in range(0, n - length + 1):
            j = i + length
            if chart[i][j]:
                span = " ".join(words[i:j])
                print(f"  chart[{i}][{j}]  \"{span}\"".ljust(38)
                      + "-> {" + ", ".join(sorted(chart[i][j])) + "}")


print_chart_grid(chart, words)
print_chart_spans(chart, words)

CKY chart -- chart[i][j] = labels spanning words[i:j]

           j=1     j=2     j=3     j=4     j=5     
           the     man     saw     the     dog     
---------------------------------------------------
i=0 (the)  {Det}   {NP}    -       -       {S}     
i=1 (man)          {N}     -       -       -       
i=2 (saw)                  {V}     -       {VP}    
i=3 (the)                          {Det}   {NP}    
i=4 (dog)                                  {N}     

Non-empty cells (by increasing span length):

  chart[0][1]  "the"                  -> {Det}
  chart[1][2]  "man"                  -> {N}
  chart[2][3]  "saw"                  -> {V}
  chart[3][4]  "the"                  -> {Det}
  chart[4][5]  "dog"                  -> {N}
  chart[0][2]  "the man"              -> {NP}
  chart[3][5]  "the dog"              -> {NP}
  chart[2][5]  "saw the dog"          -> {VP}
  chart[0][5]  "the man saw the dog"  -> {S}


### Step 5 — Recover the parse tree(s) from the back-pointers

CKY as written above is a *recogniser*. Because every cell also stores how it was built,
we can walk the back-pointers recursively and rebuild each parse tree. If a cell holds more
than one back-pointer for the same label, the sentence is ambiguous and this function returns
one tree per derivation.

In [7]:
def build_trees(chart, i, j, symbol, words):
    """Recursively rebuild every parse tree for `symbol` spanning words[i:j]."""
    trees = []
    for entry in chart[i][j].get(symbol, []):
        if isinstance(entry, str):                     # lexical entry: A -> 'word'
            trees.append(Tree(symbol, [entry]))
        else:                                          # back-pointer: A -> B C at split k
            k, b, c = entry
            for left in build_trees(chart, i, k, b, words):
                for right in build_trees(chart, k, j, c, words):
                    trees.append(Tree(symbol, [left, right]))
    return trees


cky_trees = build_trees(chart, 0, len(words), cky_grammar.start().symbol(), words)

print("Number of parse trees found by CKY:", len(cky_trees))
for t in cky_trees:
    print()
    t.pretty_print()
    print(t)

Number of parse trees found by CKY: 1

             S             
      _______|___           
     |           VP        
     |        ___|___       
     NP      |       NP    
  ___|___    |    ___|___   
Det      N   V  Det      N 
 |       |   |   |       |  
the     man saw the     dog

(S (NP (Det the) (N man)) (VP (V saw) (NP (Det the) (N dog))))


### Step 6 — Sanity check: a sentence that should be rejected

A parser that says "yes" to everything is useless, so we also feed it an ungrammatical string.
`"the saw man dog"` has no split that produces `S`, so `chart[0][n]` does not contain `S`.

In [8]:
bad_sentence = "the saw man dog"
bad_words = bad_sentence.split()
bad_chart, bad_accepted = cky_parse(bad_words, cky_grammar)

print("Sentence :", bad_sentence)
print("Accepted ?", bad_accepted)
print()
print_chart_grid(bad_chart, bad_words)

Sentence : the saw man dog
Accepted ? False

CKY chart -- chart[i][j] = labels spanning words[i:j]

           j=1     j=2     j=3     j=4     
           the     saw     man     dog     
-------------------------------------------
i=0 (the)  {Det}   -       -       -       
i=1 (saw)          {V}     -       -       
i=2 (man)                  {N}     -       
i=3 (dog)                          {N}     


## Part 2: Comparision with NLTK's chart parser
After implementing your CKY algorithm, compare its output for the given sentence with the output of NLTK's built-in chart parser using the `cky_grammar`. This comparison will help you verify if your implementation is correct.

NLTK's `ChartParser` is a general CFG parser (it does not need CNF and uses a different
strategy internally), so it is an independent implementation to check our own against.
If both agree on

* the **number** of parse trees, and
* the **trees themselves**,

then our CKY implementation is behaving correctly on this grammar.

In [9]:
parser = ChartParser(cky_grammar)
nltk_trees = list(parser.parse(words))

print("Sentence:", sentence)
print()
print("CKY (our implementation) -> ", len(cky_trees), "tree(s)")
for t in cky_trees:
    print("   ", t)

print()
print("NLTK ChartParser         -> ", len(nltk_trees), "tree(s)")
for t in nltk_trees:
    print("   ", t)

Sentence: the man saw the dog

CKY (our implementation) ->  1 tree(s)
    (S (NP (Det the) (N man)) (VP (V saw) (NP (Det the) (N dog))))

NLTK ChartParser         ->  1 tree(s)
    (S (NP (Det the) (N man)) (VP (V saw) (NP (Det the) (N dog))))


In [10]:
# Compare the two sets of trees. Trees are compared by their bracketed string form,
# which encodes both the labels and the structure.
cky_set  = sorted(str(t) for t in cky_trees)
nltk_set = sorted(str(t) for t in nltk_trees)

print("Same number of trees :", len(cky_set) == len(nltk_set))
print("Identical trees      :", cky_set == nltk_set)
print()
if cky_set == nltk_set:
    print("MATCH - the CKY implementation agrees with NLTK's chart parser.")
else:
    print("MISMATCH")
    print("  only in CKY :", [t for t in cky_set  if t not in nltk_set])
    print("  only in NLTK:", [t for t in nltk_set if t not in cky_set])

Same number of trees : True
Identical trees      : True

MATCH - the CKY implementation agrees with NLTK's chart parser.


In [11]:
# The rejection case must agree as well: NLTK should also produce zero trees.
nltk_bad = list(parser.parse(bad_words))

print(f"'{bad_sentence}'")
print("   CKY accepted        :", bad_accepted)
print("   NLTK trees produced :", len(nltk_bad))
print("   Both reject the sentence:", (not bad_accepted) and len(nltk_bad) == 0)

'the saw man dog'
   CKY accepted        : False
   NLTK trees produced : 0
   Both reject the sentence: True


### Conclusion

* The sentence **"the man saw the dog"** *is* generated by `cky_grammar`: CKY places `S` in
  `chart[0][5]`, which is exactly the acceptance condition.
* The completed chart shows how the answer is built bottom-up —
  `the man` → `NP`, `the dog` → `NP`, `saw the dog` → `VP`, and finally `NP + VP` → `S`.
* Our CKY parser and NLTK's `ChartParser` return the **same single parse tree**, and both reject
  the ungrammatical control sentence, which confirms the implementation is correct.